# Fase 2 — Integração, LLM e infraestrutura

Verifica os contratos FastAPI, o contexto agregado, a interpretação local e o cache sem rede, chave ou servidor externo. Inspeciona o HPA opcional sem criar cluster nem prometer escalabilidade na execução local. A geração real Gemini é demonstrada nos widgets dos outros notebooks, com a API configurada pelo avaliador.

Mapa, filtros, seleção de modelos, manifesto e perguntas são acessíveis pelas interfaces dos notebooks clínicos e de rotas e pelas páginas web. Os testes automatizados em `tests` complementam a validação de comportamento.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'web' / 'app.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.web.app import app

paths = app.openapi()['paths']
for path in ('/healthz', '/readyz', '/api/otimizacao/{study_name}', '/api/interpretacao', '/api/rotas/cenario', '/api/rotas/otimizar', '/api/rotas/assistente'):
    assert path in paths, path
print('Contratos API presentes. Esta verificação não inicia servidor nem chama Gemini.')


In [ ]:
import json
import os
from unittest.mock import patch
from src.optimization.llm import build_interpretation_context, generate_interpretation
from src.optimization.cache import InterpretationCache, interpretation_key

results = json.loads((PROJECT_ROOT / 'src' / 'db' / 'outputs' / 'phase2-genetic' / 'optimization_results.json').read_text(encoding='utf-8'))
for study in results['studies'].values():
    context = build_interpretation_context(study)
    assert 'models' in context and 'future_textual_data_readiness' in context
    with patch.dict(os.environ, {}, clear=True):
        response = generate_interpretation(context)
        assert response['source'] == 'local'
    print(study['label'], response['interpretation']['summary'])

calls = []
def generated_response():
    calls.append(1)
    return {'source': 'llm', 'quality_evaluation': {'passed': True}}

cache = InterpretationCache()
key = interpretation_key(context, None)
cache.get_or_generate(key, generated_response)
cached = cache.get_or_generate(key, generated_response)
assert len(calls) == 1 and cached['cache']['hit']
print('Cache validado com resposta simulada, não gerada pelo Gemini.')


In [ ]:
manifest = json.loads((PROJECT_ROOT / 'infra' / 'autoscaling' / 'kubernetes.json').read_text(encoding='utf-8'))
resources = {item['kind']: item for item in manifest['items']}
hpa = resources['HorizontalPodAutoscaler']['spec']
assert hpa['minReplicas'] == 1 and hpa['maxReplicas'] == 4
print('HPA opcional:', hpa['minReplicas'], 'a', hpa['maxReplicas'], 'réplicas')
print('Logs HTTP: método, caminho, status e latência. Experimentos: optimization.log.')
print('Para testar escala real, use scripts/test_autoscaling.ps1 após preparar Kubernetes e Metrics Server; este notebook não implanta infraestrutura.')
